# B3 — equilibrium-augmented: testing an early-cutoff w_equilibrium schedule

**Follow-up paper, Professor Rabczuk's items 2/3.** The first 4-variant pilot (`B3_Equilibrium_Augmented_Diagnostic.ipynb`) found something more specific than a simple miscalibration: all 4 variants (pure energy, and 3 equilibrium-augmented variants spanning weight 0.03-0.12 and `n_virtual` 8-32) tracked **identically** up to iteration ~2250 -- no visible benefit or harm from the equilibrium term at that stage -- then from iteration ~2500 onward the pure-energy reference kept improving while **all three** equilibrium-augmented variants diverged upward together, regardless of weight or noise level. This rules out "weight still too large" and "residual too noisy" as the explanation; the term appears to destabilize training once it has matured past a certain point, independent of its magnitude within the tested range.

**This pilot tests the one remaining reasonable idea**: since the term looks harmless before the observed onset and harmful after, phase it out on an ABSOLUTE-iteration schedule before training reaches that regime, rather than holding it constant or re-tuning its magnitude again. `w_equilibrium_decay_start=1000, w_equilibrium_decay_end=2500` (new flags on `train_B3_equilibrium_augmented.py`, 2026-10-24): constant `0.12` up to iteration 1,000, linearly down to 0 by iteration 2,500 -- safely before the ~2500-3000 zone where the first pilot's fixed-weight variants first showed clear divergence. Default (0, 0) leaves every existing caller/checkpoint unaffected.

**Two variants, 6,000 iterations each** (longer than the first pilot's 4,000, specifically to see whether turning the term off in time actually prevents the later degradation rather than merely delaying it): a pure-energy reference (`w_equilibrium=0`, same as the first pilot's own reference, re-run here for a matching iteration count) and the decay-schedule variant. No final FEM evaluation (still a pilot, not a full run) -- if the decay variant tracks the pure-energy reference all the way to 6,000 iterations, that is strong evidence for a full 50,000-iteration attempt 3 with this schedule; if it still diverges, that would mean even an early, fully-phased-out application of this term leaves a lasting negative effect, a stronger and different finding that would argue for abandoning this specific mechanism rather than tuning it further.

In [ ]:
# =====================================================================
#  CELL -- B3 equilibrium-augmented: SHORT (6,000-iteration) pilot
#  testing an early-cutoff w_equilibrium DECAY SCHEDULE against a
#  pure-energy reference. See the markdown cell above for the full
#  rationale (follow-up to B3_Equilibrium_Augmented_Diagnostic.ipynb's
#  own 4-variant pilot).
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

from omar_pfem.train_B3_equilibrium_augmented import get_eq_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')
R = '/content/drive/MyDrive/pfem_run'
N_ITERS = 6000

VARIANTS = [
    ('baseline_w0.0_6k', dict(w_equilibrium='0.0', decay_start='0', decay_end='0')),
    ('decay_1000_2500',  dict(w_equilibrium='0.12', decay_start='1000', decay_end='2500')),
]

n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nPilot: {DEFAULT_RESOLUTION} -> {n_elem} elements, {N_ITERS} iterations per variant, '
      f'{len(VARIANTS)} variants, no final FEM evaluation (still a pilot).')

summary = []
for name, cfg in VARIANTS:
    print(f'\n{"=" * 90}')
    print(f'=== Variant: {name} ({cfg}) ===')
    OUTPUT_DIR = f'{R}/b3_equilibrium_decay_pilot_{name}'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = get_eq_args([
        '--n_iters', str(N_ITERS),
        '--batch_size', '8',
        '--log_every', '250',
        '--ckpt_every', str(N_ITERS),
        '--output_dir', OUTPUT_DIR,
        '--w_equilibrium', cfg['w_equilibrium'],
        '--n_virtual', '8',
        '--w_equilibrium_decay_start', cfg['decay_start'],
        '--w_equilibrium_decay_end', cfg['decay_end'],
    ])
    torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    model = train(args, device)
    elapsed = time.time() - t0
    print(f'{name}: finished in {elapsed:.1f}s ({elapsed/N_ITERS:.4f}s/iteration).')
    summary.append((name, cfg, elapsed))

print(f'\n{"=" * 90}')
print('=== Pilot summary -- compare the printed `energy` trend above for both variants, '
      'especially from iteration ~2500 onward (where the first pilot\'s fixed-weight '
      'variants diverged): if decay_1000_2500 tracks baseline_w0.0_6k all the way to 6,000, '
      'that supports a full 50,000-iteration attempt 3 with this schedule. ===')
for name, cfg, elapsed in summary:
    print(f'  {name}: {cfg}, {elapsed:.1f}s total')

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(f'{R}/b3_equilibrium_decay_pilot', kind='b3_equilibrium_augmented_diagnostic',
                    args={'n_iters': N_ITERS, 'variants': [v[0] for v in VARIANTS]},
                    started_at=_started,
                    results={'elapsed_total_s': time.time() - _started},
                    outputs=[f'{R}/b3_equilibrium_decay_pilot_{v[0]}' for v in VARIANTS],
                    notes="Follow-up to B3_Equilibrium_Augmented_Diagnostic.ipynb's 4-variant "
                          "pilot, which found the equilibrium term destabilizes training from "
                          "iteration ~2500 onward regardless of weight (0.03-0.12) or "
                          "n_virtual (8/32). This pilot tests an absolute-iteration decay "
                          "schedule (w_equilibrium_decay_start/end, new 2026-10-24) that phases "
                          "the term out before that onset, over 6,000 iterations (longer than "
                          "the first pilot) to check whether an early, fully-phased-out "
                          "application avoids the later degradation or merely delays it.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back -- especially the `energy` values '
      'at matched iterations for both variants from 2500 onward.')
